# Iris 분류 with Pipeline

00_iris의 문제(데이터 누수, 스케일링 미적용)를 Pipeline으로 해결한 버전.
스케일러 + 모델을 하나로 묶어, 분할 후 train에만 fit 되도록 구조적으로 보장한다.

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
print(X.shape, y.shape)
print(iris.target_names)

(150, 4) (150,)
['setosa' 'versicolor' 'virginica']


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)
print(X_train.shape, X_test.shape)
print(np.unique(y_train, return_counts=True))

(105, 4) (45, 4)
(array([0, 1, 2]), array([35, 35, 35]))


## Pipeline 만들기

StandardScaler + KNN을 하나로 묶기
pipe.fit(X_train)을 하면 스케일러는 train으로만 fit → KNN 학습까지 한 번에. 
pipe.predict(X_test)는 test에 같은 기준으로 transform 후 예측.
→ 데이터 누수 불가능

In [5]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print(accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))

0.9111111111111111
[[15  0  0]
 [ 0 15  0]
 [ 0  4 11]]


## 교차검증으로 k 고르기

테스트 세트로 k를 고르면 누수
train 안에서 cross_val_score로 k를 고르고, test는 마지막에 한번만

In [11]:
from sklearn.model_selection import cross_val_score

for k in range(1, 21):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5)
    print(k, round(scores.mean(), 4))

[0.95238095 0.95238095 1.         0.9047619  0.95238095]
1 0.9524
[0.95238095 0.9047619  1.         0.9047619  0.9047619 ]
2 0.9333
[1.        0.9047619 1.        0.9047619 1.       ]
3 0.9619
[0.95238095 0.9047619  1.         0.9047619  1.        ]
4 0.9524
[0.95238095 0.95238095 1.         0.9047619  1.        ]
5 0.9619
[0.9047619  0.95238095 1.         0.9047619  0.95238095]
6 0.9429
[0.95238095 0.9047619  1.         0.9047619  1.        ]
7 0.9524
[0.95238095 0.9047619  1.         0.9047619  1.        ]
8 0.9524
[0.95238095 0.9047619  1.         0.95238095 1.        ]
9 0.9619
[0.95238095 0.9047619  1.         0.95238095 1.        ]
10 0.9619
[0.95238095 0.9047619  1.         0.95238095 1.        ]
11 0.9619
[0.95238095 0.95238095 1.         0.95238095 1.        ]
12 0.9714
[0.95238095 0.9047619  1.         0.95238095 1.        ]
13 0.9619
[0.95238095 0.9047619  1.         1.         1.        ]
14 0.9714
[0.95238095 0.9047619  1.         1.         0.95238095]
15 0.9619
[0.952380

In [10]:
best_k = 12

final = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
])
final.fit(X_train, y_train)
final_pred = final.predict(X_test)

print("최종 정확도:", accuracy_score(y_test, final_pred))
print(confusion_matrix(y_test, final_pred))

최종 정확도: 0.9555555555555556
[[15  0  0]
 [ 0 15  0]
 [ 0  2 13]]
